# Function 3: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('../../data/initial_data/function_3/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.754364, 0.233177, 0.303208],
    [0.312229, 0.060777, 0.000904],
    [0.5, 0.5, 0.5],
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (15, 3)
After: (17, 3)
[[1.71525207e-01 3.43916870e-01 2.48737201e-01]
 [2.42114461e-01 6.44074270e-01 2.72432809e-01]
 [5.34905720e-01 3.98500915e-01 1.73388729e-01]
 [4.92581415e-01 6.11593188e-01 3.40176386e-01]
 [1.34621666e-01 2.19917240e-01 4.58206220e-01]
 [3.45523271e-01 9.41359831e-01 2.69363479e-01]
 [1.51836632e-01 4.39990619e-01 9.90881867e-01]
 [6.45502835e-01 3.97142940e-01 9.19771338e-01]
 [7.46911945e-01 2.84196309e-01 2.26299855e-01]
 [1.70476994e-01 6.97032401e-01 1.49169434e-01]
 [2.20549337e-01 2.97825244e-01 3.43555344e-01]
 [6.66013659e-01 6.71985151e-01 2.46295297e-01]
 [4.68089497e-02 2.31360241e-01 7.70617592e-01]
 [6.00097282e-01 7.25135725e-01 6.60886415e-02]
 [9.65994849e-01 8.61119690e-01 5.66829131e-01]
 [7.54364000e-01 2.33177000e-01 3.03208000e-01]
 [3.12229000e-01 6.07770000e-02 9.04000000e-04]]


In [ ]:
output_data = np.load('../../data/initial_data/function_3/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.09200841551496666,
    -0.18083748652026374,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


NameError: name 'np' is not defined

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5]])
actual_output = -0.015979341188442648

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 3
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 3, dimension d=3
Data shape: (18, 3) (18,)
Current best observed y: -0.3989255131463011
Current best x: [0.15183663 0.43999062 0.99088187]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,y,log_abs_y,rank_min
6,0.151837,0.439991,0.990882,-0.398926,-0.918981,1
16,0.312229,0.060777,0.000904,-0.180837,-1.710157,2
8,0.746912,0.284196,0.226300,-0.131461,-2.029048,3
12,0.046809,0.231360,0.770618,-0.118048,-2.136662,4
7,0.645503,0.397143,0.919771,-0.113869,-2.172711,5
0,0.171525,0.343917,0.248737,-0.112122,-2.188166,6
2,0.534906,0.398501,0.173389,-0.111415,-2.194496,7
5,0.345523,0.941360,0.269363,-0.110621,-2.201646,8
11,0.666014,0.671985,0.246295,-0.105965,-2.244646,9
9,0.170477,0.697032,0.149169,-0.094190,-2.362446,10


## Classification framing: good vs bad outputs

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Define "good" as the best quartile of observed outputs.
# For a very small dataset this gives enough positive labels to fit a classifier.
good_threshold = np.quantile(output_data, 0.25)
good_label = (output_data <= good_threshold).astype(int)

class_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
class_df["y"] = output_data
class_df["good_label"] = good_label
class_df["distance_to_threshold"] = np.abs(output_data - good_threshold)
class_df["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
class_df = class_df.sort_values("distance_to_threshold")

print(f"Good/bad threshold: y <= {good_threshold:.6e}")
display(class_df)

print("Support-vector-like observed points:")
display(class_df.head(min(5, len(class_df))))


Good/bad threshold: y <= -1.134319e-01


,x1,x2,x3,y,good_label,distance_to_threshold,log_abs_y
7,0.645503,0.397143,0.919771,-0.113869,1,0.000437,-2.172711
0,0.171525,0.343917,0.248737,-0.112122,0,0.001310,-2.188166
2,0.534906,0.398501,0.173389,-0.111415,0,0.002017,-2.194496
5,0.345523,0.941360,0.269363,-0.110621,0,0.002811,-2.201646
12,0.046809,0.231360,0.770618,-0.118048,1,0.004616,-2.136662
11,0.666014,0.671985,0.246295,-0.105965,0,0.007467,-2.244646
8,0.746912,0.284196,0.226300,-0.131461,1,0.018029,-2.029048
9,0.170477,0.697032,0.149169,-0.094190,0,0.019242,-2.362446
15,0.754364,0.233177,0.303208,-0.092008,0,0.021424,-2.385875
1,0.242114,0.644074,0.272433,-0.087963,0,0.025469,-2.430841


Support-vector-like observed points:


,x1,x2,x3,y,good_label,distance_to_threshold,log_abs_y
7,0.645503,0.397143,0.919771,-0.113869,1,0.000437,-2.172711
0,0.171525,0.343917,0.248737,-0.112122,0,0.001310,-2.188166
2,0.534906,0.398501,0.173389,-0.111415,0,0.002017,-2.194496
5,0.345523,0.941360,0.269363,-0.110621,0,0.002811,-2.201646
12,0.046809,0.231360,0.770618,-0.118048,1,0.004616,-2.136662


In [6]:
X = input_data.copy()
y_cls = good_label
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if len(np.unique(y_cls)) < 2:
    print("Only one class is present, so logistic/SVM classifiers cannot be fitted yet.")
else:
    log_reg = LogisticRegression(class_weight="balanced", random_state=0)
    svm_linear = SVC(kernel="linear", class_weight="balanced", probability=True, random_state=0)
    svm_rbf = SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", probability=True, random_state=0)

    models = {"logistic": log_reg, "linear_svm": svm_linear, "rbf_svm": svm_rbf}
    for name, model in models.items():
        model.fit(X_scaled, y_cls)
        pred = model.predict(X_scaled)
        print("\n", name)
        print("Confusion matrix:\n", confusion_matrix(y_cls, pred))
        print(classification_report(y_cls, pred, zero_division=0))



 logistic
Confusion matrix:
 [[10  3]
 [ 0  5]]
              precision    recall  f1-score   support

           0       1.00      0.77      0.87        13
           1       0.62      1.00      0.77         5

    accuracy                           0.83        18
   macro avg       0.81      0.88      0.82        18
weighted avg       0.90      0.83      0.84        18


 linear_svm
Confusion matrix:
 [[9 4]
 [0 5]]
              precision    recall  f1-score   support

           0       1.00      0.69      0.82        13
           1       0.56      1.00      0.71         5

    accuracy                           0.78        18
   macro avg       0.78      0.85      0.77        18
weighted avg       0.88      0.78      0.79        18


 rbf_svm
Confusion matrix:
 [[12  1]
 [ 0  5]]
              precision    recall  f1-score   support

           0       1.00      0.92      0.96        13
           1       0.83      1.00      0.91         5

    accuracy                          

In [7]:
# Boundary / uncertainty search: where classifier is closest to p(good)=0.5.
if len(np.unique(y_cls)) >= 2:
    rng = np.random.default_rng(1)
    candidates = rng.random((30000 if d <= 4 else 60000, d))
    cand_scaled = scaler.transform(candidates)

    rows = []
    for name, model in models.items():
        prob_good = model.predict_proba(cand_scaled)[:, 1]
        uncertainty = np.abs(prob_good - 0.5)
        idx = np.argsort(uncertainty)[:10]
        tmp = pd.DataFrame(candidates[idx], columns=[f"x{i+1}" for i in range(d)])
        tmp["model"] = name
        tmp["p_good"] = prob_good[idx]
        tmp["boundary_score_abs_p_minus_0.5"] = uncertainty[idx]
        rows.append(tmp)

    boundary_points = pd.concat(rows, ignore_index=True)
    display(boundary_points.sort_values("boundary_score_abs_p_minus_0.5").head(20))

    candidate = boundary_points.sort_values("boundary_score_abs_p_minus_0.5").iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
    print("Most boundary-like candidate:", np.round(candidate, 6))
    print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(candidate)))


,x1,x2,x3,model,p_good,boundary_score_abs_p_minus_0.5
14,0.615961,0.210426,0.128178,linear_svm,0.5,0.0
27,0.585947,0.756696,0.952739,rbf_svm,0.5,0.0
26,0.996015,0.469934,0.826598,rbf_svm,0.5,0.0
25,0.812174,0.382628,0.496906,rbf_svm,0.5,0.0
24,0.948339,0.445493,0.565144,rbf_svm,0.5,0.0
23,0.097642,0.034823,0.659704,rbf_svm,0.5,0.0
22,0.542223,0.639111,0.827109,rbf_svm,0.5,0.0
21,0.764045,0.448545,0.286602,rbf_svm,0.5,0.0
20,0.614092,0.746810,0.947072,rbf_svm,0.5,0.0
19,0.570545,0.206161,0.148615,linear_svm,0.5,0.0


Most boundary-like candidate: [0.615961 0.210426 0.128178]
Portal format: x1=0.615961, x2=0.210426, x3=0.128178


In [10]:
# Optional 2D plot
if d == 2 and len(np.unique(y_cls)) >= 2:
    grid_res = 200
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    for name, model in models.items():
        prob = model.predict_proba(grid_scaled)[:, 1].reshape(grid_res, grid_res)
        plt.figure(figsize=(7, 6))
        cf = plt.contourf(xx, yy, prob, levels=40)
        plt.colorbar(cf, label="Predicted P(good)")
        plt.contour(xx, yy, prob, levels=[0.5], linewidths=2)
        plt.scatter(input_data[:, 0], input_data[:, 1], c=good_label, edgecolors="black", s=80)
        plt.xlabel("x1"); plt.ylabel("x2"); plt.title(f"Good/bad boundary: {name}")
        plt.show()
